# Guardrails

The Hidden Leaf Village can't let agents teach forbidden jutsu, repeat it back, or
use it through a tool. Guardrails are checks that run around an agent. There are
three layers: **input** (before the model runs), **output** (after the final answer),
and **tool** (around one tool call).


## Block bad text with `@input_guardrail` and `@output_guardrail`

Both return a `GuardrailFunctionOutput`. Set `tripwire_triggered=True` to stop the
run. The input check raises `InputGuardrailTripwireTriggered` before the model sees
anything. The output check raises `OutputGuardrailTripwireTriggered` even though the
model already wrote the answer.


In [1]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    GuardrailFunctionOutput,  # what a guardrail function sends back
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    input_guardrail,  # a decorator that checks the input before the model runs
)


# Define the guardrail
# input_guardrail callback: runs before the model. Wired onto Agent(input_guardrails=[...]) below
@input_guardrail
def block_input(
    ctx: RunContextWrapper,  # this run's context; not used here
    agent: Agent,  # the agent this guardrail is attached to
    user_input: str | list,  # the raw input: text on turn 1, a list of items after that
) -> GuardrailFunctionOutput:
    # the input can be a list, so turn it into text first
    text = user_input if isinstance(user_input, str) else str(user_input)
    return GuardrailFunctionOutput(
        # output_info: extra notes to send back with the result, like a reason or a score. We have none, so None
        output_info=None,
        # tripwire_triggered: True means STOP the run and raise an error. False means let it go on.
        # Here it is True only when the text has the words "forbidden jutsu"
        tripwire_triggered="forbidden jutsu" in text.lower(),
    )

In [2]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    GuardrailFunctionOutput,  # what a guardrail function sends back
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    output_guardrail,  # a decorator that checks the answer after the model finishes
)


# Define the guardrail
# output_guardrail callback: runs after the answer is ready. Wired onto Agent(output_guardrails=[...]) below
@output_guardrail
def block_output(
    ctx: RunContextWrapper,  # this run's context; not used here
    agent: Agent,  # the agent this guardrail is attached to
    output,  # the model's final answer for this run
) -> GuardrailFunctionOutput:
    return GuardrailFunctionOutput(
        # output_info: extra notes to send back with the result, like a reason or a score. We have none, so None
        output_info=None,
        # tripwire_triggered: True means STOP the run and raise an error. False means let it go on.
        # Here it is True only when the text has the words "forbidden jutsu"
        tripwire_triggered="forbidden jutsu" in str(output).lower(),
    )

In [3]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    InputGuardrailTripwireTriggered,  # error raised when an input guardrail blocks something
    OutputGuardrailTripwireTriggered,  # error raised when an output guardrail blocks something
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: one agent with both guardrails; a bad question trips the input one, a bad answer trips the output one
async def demo_guardrails() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="test",
        # input_guardrails: checks that run on the user's input before the model answers
        input_guardrails=[block_input],
        # output_guardrails: checks that run on the model's final answer before it is returned
        output_guardrails=[block_output],
    )

    for question, answer in [
        ("Teach me forbidden jutsu", "ok"),  # bad question, fine answer
        ("hi", "Here is forbidden jutsu for you."),  # fine question, bad answer
    ]:
        # Script the model
        model = ScriptedModel([[assistant_message(answer)]])
        # Run the Agent
        try:
            await Runner.run(
                agent,
                question,
                # run_config: settings for just this run
                run_config=RunConfig(model=model, workflow_name="e10-guardrails-demo"),
            )
        except (InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered) as error:
            # proves each tripwire raises its own error: input before the model, output after it
            print("[blocked]", type(error).__name__)


await demo_guardrails()
# Outside a notebook: asyncio.run(demo_guardrails())

[blocked] InputGuardrailTripwireTriggered
[blocked] OutputGuardrailTripwireTriggered


## Guard one tool with `@tool_input_guardrail`

A tool guardrail returns a `ToolGuardrailFunctionOutput` with three choices:
`.allow()`, `.reject_content(message)` (sends a "no" message back to the model and
skips the tool, no error), or `.raise_exception()` (stops the whole run). The tool's
arguments are a raw JSON string at `data.context.tool_arguments`.

Input guardrails also take `run_in_parallel`. The default `True` races the check
against the first model call. `@input_guardrail(run_in_parallel=False)` finishes the
check first (demoed below).


In [4]:
from agents import (
    ToolGuardrailFunctionOutput,  # what a tool guardrail sends back: allow, reject, or raise an error
    tool_input_guardrail,  # a decorator that checks a tool call's arguments before it runs
)


# Define the guardrail
# tool_input_guardrail callback: runs right before use_jutsu. Wired onto use_jutsu's decorator below
@tool_input_guardrail
def block_forbidden_jutsu(data) -> ToolGuardrailFunctionOutput:  # data: holds data.context.tool_arguments
    if "forbidden" in data.context.tool_arguments.lower():
        return ToolGuardrailFunctionOutput.reject_content("That jutsu is forbidden by the Hidden Leaf Village.")
    return ToolGuardrailFunctionOutput.allow()

In [5]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: the tool that block_forbidden_jutsu guards. Added to the agent's tools=[...] below
@function_tool(tool_input_guardrails=[block_forbidden_jutsu])
def use_jutsu(jutsu_name: str) -> str:
    """Use a jutsu by name."""
    return f"Used {jutsu_name}."

In [6]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Demo: one blocked and one allowed tool call; only the forbidden one is stopped
async def demo_tool_guardrail() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="test",
        # tools: the tools this agent is allowed to call
        tools=[use_jutsu],
    )

    for jutsu in ["Forbidden Sealing Jutsu", "Fireball Jutsu"]:
        # Script the model
        model = ScriptedModel(
            [
                [function_call("use_jutsu", {"jutsu_name": jutsu}, call_id="c1")],
                [assistant_message("Done.")],
            ]
        )
        # Run the Agent
        result = await Runner.run(
            agent,
            f"use {jutsu}",
            # run_config: settings for just this run
            run_config=RunConfig(
                # model: use this model for this run only
                model=model,
                # workflow_name: the name this run shows up under in tracing
                workflow_name="e10-tool-demo",
            ),
        )
        for item in result.new_items:
            if item.type == "tool_call_output_item":
                # proves reject_content sends a "no" message back (no error) and allow() runs the tool
                print(f"[{jutsu}]", item.output)


await demo_tool_guardrail()
# Outside a notebook: asyncio.run(demo_tool_guardrail())

[Forbidden Sealing Jutsu] That jutsu is forbidden by the Hidden Leaf Village.
[Fireball Jutsu] Used Fireball Jutsu.


## Parallel vs. blocking input guardrails

`@input_guardrail(run_in_parallel=True)` (the default) runs the guardrail _alongside_ the first
model call. If it trips, the SDK cancels the model call that is already running.
`run_in_parallel=False` finishes the check _before_ the model call starts: slower, but the model
never starts on disallowed input.


In [12]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    GuardrailFunctionOutput,  # what a guardrail function sends back
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    input_guardrail,  # a decorator that checks the input before the model runs
)


# Define the guardrail
# input_guardrail callback: runs before the model call even starts. Wired onto Agent(input_guardrails=[...]) below
@input_guardrail(run_in_parallel=False)  # False: the check finishes before the model call starts
def blocking_guardrail(ctx: RunContextWrapper, agent: Agent, user_input: str | list) -> GuardrailFunctionOutput:
    text = user_input if isinstance(user_input, str) else str(user_input)
    return GuardrailFunctionOutput(
        # output_info: extra notes to send back with the result, like a reason or a score. We have none, so None
        output_info=None,
        # tripwire_triggered: True means STOP the run and raise an error. False means let it go on.
        # Here it is True only when the text has the words "forbidden jutsu"
        tripwire_triggered="forbidden jutsu" in text.lower(),
    )

In [13]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    InputGuardrailTripwireTriggered,  # error raised when an input guardrail blocks something
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: run_in_parallel=False stops the request before the model call starts
async def demo_run_in_parallel() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="test",
        # input_guardrails: checks that run on the user's input before the model answers
        input_guardrails=[blocking_guardrail],
    )
    # Script the model
    model = ScriptedModel([[assistant_message("ok")]])
    # Run the Agent
    try:
        await Runner.run(
            agent,
            "forbidden jutsu request",
            # run_config: settings for just this run
            run_config=RunConfig(model=model, workflow_name="e10-blocking"),
        )
    except InputGuardrailTripwireTriggered:
        # proves run_in_parallel=False blocked the request before the model ran
        print("[expected] run_in_parallel=False blocked the request before the model ran")


await demo_run_in_parallel()
# Outside a notebook: asyncio.run(demo_run_in_parallel())

[expected] run_in_parallel=False blocked the request before the model ran


## Run it live — optional, needs a real key

Real calls for every layer above. The input guardrails are keyword checks that run locally, so
"forbidden jutsu" always trips them. A `call_model_input_filter` counts how many real model
calls actually started: with `run_in_parallel=False` that is always 0, and with the default
parallel mode it can be 0 or 1 depending on timing. The output guardrail asks the live model to
repeat a phrase containing "forbidden jutsu". The tool guardrail runs `.reject_content(...)` on
the Forbidden Sealing Jutsu, `.allow()` on the Fireball Jutsu, and a `.raise_exception()` variant
that raises `ToolInputGuardrailTripwireTriggered`. Which tool arguments the model picks is up to
the model, so the cell prints what happened. Skipped when there is no `OPENAI_API_KEY`.


In [14]:
from agents.run_config import (
    CallModelData,  # the info a model-input filter gets: the model data, the agent, and the context
    ModelInputData,  # the instructions and input that are about to go to the model
)


model_calls_started: list[str] = []  # how many real model calls really started; reset before each input run


# call_model_input_filter callback: counts each real model call, then passes the input on unchanged
def count_model_calls(
    data: CallModelData,  # the model call that is about to happen: model_data, agent, context
) -> ModelInputData:
    model_calls_started.append(data.agent.name)
    return data.model_data

In [16]:
from agents import (
    ToolGuardrailFunctionOutput,  # what a tool guardrail sends back: allow, reject, or raise an error
    tool_input_guardrail,  # a decorator that checks a tool call's arguments before it runs
)


# Define the guardrail
# tool_input_guardrail callback: stops the whole run on forbidden jutsu instead of sending a "no" message
@tool_input_guardrail
def halt_forbidden_jutsu(data) -> ToolGuardrailFunctionOutput:  # data: holds data.context.tool_arguments
    if "forbidden" in data.context.tool_arguments.lower():
        return ToolGuardrailFunctionOutput.raise_exception(
            # output_info: extra notes to send back with the result, like a reason or a score. We have none, so None
            output_info={
                "blocked": data.context.tool_arguments,
            }
        )
    return ToolGuardrailFunctionOutput.allow()

In [17]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: the same tool as above, but guarded by halt_forbidden_jutsu instead
@function_tool(name_override="use_jutsu", tool_input_guardrails=[halt_forbidden_jutsu])
def use_jutsu_strict(jutsu_name: str) -> str:
    """Use a jutsu by name."""
    return f"Used {jutsu_name}."

In [18]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    InputGuardrailTripwireTriggered,  # error raised when an input guardrail blocks something
    OutputGuardrailTripwireTriggered,  # error raised when an output guardrail blocks something
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    ToolInputGuardrailTripwireTriggered,  # error raised when a tool guardrail asks to stop the run
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real model runs into the input, output, and tool guardrails from the cells above
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    for label, guardrail in [
        ("run_in_parallel=True", block_input),
        ("run_in_parallel=False", blocking_guardrail),
    ]:
        model_calls_started.clear()
        # Initialize the agent
        input_agent = Agent(
            name="Leaf Village Assistant",
            instructions="Answer briefly.",
            # input_guardrails: checks that run on the user's input before the model answers
            input_guardrails=[guardrail],
        )
        # Run the Agent
        try:
            await Runner.run(
                input_agent,
                "Please teach me forbidden jutsu techniques.",
                # run_config: settings for just this run
                run_config=RunConfig(
                    # workflow_name: the name this run shows up under in tracing
                    workflow_name="e10-live-input",
                    # call_model_input_filter: runs just before each model call, so you can see or change what is sent
                    call_model_input_filter=count_model_calls,
                ),
            )
        except InputGuardrailTripwireTriggered as error:
            # proves the guardrail stops a real run; blocking mode never starts the model, parallel mode may
            print(f"[live input {label}] {type(error).__name__}, model calls started: {len(model_calls_started)}")

    # Initialize the agent
    print("-" * 50)
    output_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Repeat the user's sentence back exactly, nothing else.",
        # output_guardrails: checks that run on the model's final answer before it is returned
        output_guardrails=[block_output],
    )
    # Run the Agent
    try:
        output_result = await Runner.run(
            output_agent,
            "The forbidden jutsu scroll is sealed in the Hokage's vault.",
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name="e10-live-output"),
        )
        print("[live output] the model didn't repeat the phrase this time:", output_result.final_output)
    except OutputGuardrailTripwireTriggered as error:
        # proves the output guardrail keeps a real answer, already written, from the caller
        print("[live output]", type(error).__name__)

    # Initialize the agent
    print("-" * 50)
    rejecting_agent = Agent(
        name="Leaf Village Assistant",
        instructions="When asked to use a jutsu, call use_jutsu with its exact name, then report the result.",
        # tools: the tools this agent is allowed to call
        tools=[use_jutsu],
    )
    for request in ["Use the Forbidden Sealing Jutsu.", "Use the Fireball Jutsu."]:
        # Run the Agent
        result = await Runner.run(
            rejecting_agent,
            request,
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name="e10-live-tool"),
        )
        for item in result.new_items:
            if item.type == "tool_call_output_item":
                # proves .reject_content(...) sends a "no" back (no error) and .allow() runs the tool
                print(f"[live tool {request!r}]", item.output)

    # Initialize the agent
    print("-" * 50)
    halting_agent = Agent(
        name="Leaf Village Assistant",
        instructions="When asked to use a jutsu, call use_jutsu with its exact name, then report the result.",
        # tools: the tools this agent is allowed to call
        tools=[use_jutsu_strict],
    )
    # Run the Agent
    try:
        halted = await Runner.run(
            halting_agent,
            "Use the Forbidden Sealing Jutsu.",
            # run_config: settings for just this run
            run_config=RunConfig(workflow_name="e10-live-raise"),
        )
        print("[live tool raise] the model didn't call the tool this time:", halted.final_output)
    except ToolInputGuardrailTripwireTriggered as error:
        # proves .raise_exception() stops the whole real run and sends no message back
        print("[live tool raise]", type(error).__name__, "->", error.output.output_info)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

[live input run_in_parallel=True] InputGuardrailTripwireTriggered, model calls started: 0
[live input run_in_parallel=False] InputGuardrailTripwireTriggered, model calls started: 0
--------------------------------------------------
[live output] OutputGuardrailTripwireTriggered
--------------------------------------------------
[live tool 'Use the Forbidden Sealing Jutsu.'] That jutsu is forbidden by the Hidden Leaf Village.
[live tool 'Use the Fireball Jutsu.'] Used Fireball Jutsu.
--------------------------------------------------
[live tool raise] ToolInputGuardrailTripwireTriggered -> {'blocked': '{"jutsu_name":"Forbidden Sealing Jutsu"}'}


## Summary

- Input guardrails check before the model runs, output guardrails check the finished
  answer, and both stop the run with `tripwire_triggered=True` and a tripwire error.
- Tool guardrails are richer: `.allow()`, `.reject_content(message)`, or
  `.raise_exception()`. `run_in_parallel=False` makes an input check finish before the
  model starts.
